In [1]:
import pandas as pd
from utils.ff_functions import run_GRS

In [2]:
# Reading dfs
df_SIZE = pd.read_parquet("../data/processed/df_SIZE.parquet")
df_INV = pd.read_parquet("../data/processed/df_INV.parquet")
df_OP = pd.read_parquet("../data/processed/df_OP.parquet")
factors_df = pd.read_parquet("../data/processed/factors_df.parquet")

# GRS Test on CAPM
---
The Gibbons-Ross-Shanken (GRS) test is testing the joint null hypothesis that the intercepts (α's) from the CAPM
time-series regressions are simultaneously zero across all 25 portfolios:

$$H_0: \alpha_1 = \alpha_2 = ... = \alpha_{25} = 0$$

Rejection indicates that the market factor alone cannot fully explain the cross-sectional
variation in portfolio returns.

In [3]:
# Factors for FF5 GRS
facts_capm = ["Mkt-RF"]
# Portfolio dfs
df_dict = {"SIZE" : df_SIZE,
           "INV" : df_INV,
           "OP" : df_OP}

grs_rows = []

for name, df in df_dict.items():
    portfolio_cols = [c for c in df.columns if c != 'Date']

    F_stat, pVal, _, _ = run_GRS(df, factors_df, portfolio_cols, facts_capm)

    grs_rows.append({
        "set": name,
        "GRS_F": F_stat,
        "GRS_p": round(pVal, 4),
        "n_portfolios": len(portfolio_cols),
    })

capm_grs_df = pd.DataFrame(grs_rows).set_index("set")
display(capm_grs_df)

,GRS_F,GRS_p,n_portfolios
set,,,
SIZE,4.452366,0.0000,25
INV,5.513548,0.0000,25
OP,2.413719,0.0002,25


### Interpretation
The GRS F-statistic is large and significant across all three portfolio sets, with the investment
portfolios showing the highest joint pricing error (F = 5.51). The null hypothesis that all alphas
are jointly zero is decisively rejected at all conventional significance levels.

# GRS Test on FF3
---

The GRS test is now using the residuals from the FF3 model. The null hypothesis is that the three-factor
model leaves no systematic pricing error across all 25 portfolios:

$$H_0: \alpha_1 = \alpha_2 = ... = \alpha_{25} = 0$$

If we reject the null, after controlling for size and value, we have evidence that additional risk
dimensions remain unpriced by FF3.

In [4]:
# Factors for FF5 GRS
facts_ff3 = ["Mkt-RF", "SMB", "HML"]
# Portfolio dfs
df_dict = {"SIZE" : df_SIZE,
           "INV" : df_INV,
           "OP" : df_OP}

grs_rows = []

for name, df in df_dict.items():
    # Exclude date from cols
    portfolio_cols = [c for c in df.columns if c != 'Date']

    F_stat, pVal, _, _ = run_GRS(df, factors_df, portfolio_cols, facts_ff3)

    grs_rows.append({
        "set": name,
        "GRS_F": F_stat,
        "GRS_p": round(pVal, 4),
        "n_portfolios": len(portfolio_cols),
    })

ff3_grs_df = pd.DataFrame(grs_rows).set_index("set")
display(ff3_grs_df)

,GRS_F,GRS_p,n_portfolios
set,,,
SIZE,3.545125,0.0000,25
INV,4.685143,0.0000,25
OP,2.541858,0.0001,25


### Interpretation
The GRS F-statistic falls relative to CAPM across all portfolio sets (F = 3.55, 4.69, 2.54),
indicating that SMB and HML meaningfully reduce joint pricing error. However, the null is
still rejected across all sets. The model performs relatively worse on the investment and
profitability sorts, as these dimensions are not directly captured by FF3 factors.

# GRS Test on FF5
---

The GRS test is applied to FF5 residuals. The null hypothesis is that the five-factor
model fully prices all 25 portfolios with zero systematic error:

$$H_0: \alpha_1 = \alpha_2 = ... = \alpha_{25} = 0$$

If we reject the null while controlling for market, size, value, profitability and investment, we can conclude that even
with these additional risk proxies we are still mispricing at least some of the portfolio excess returns.

In [6]:
# Factors for FF5 GRS
facts_ff5 = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
# Portfolio dfs
df_dict = {"SIZE" : df_SIZE,
           "INV" : df_INV,
           "OP" : df_OP}

grs_rows = []

for name, df in df_dict.items():
    # Exclude Date from cols
    portfolio_cols = [c for c in df.columns if c != 'Date']

    F_stat, pVal, _, _ = run_GRS(df, factors_df, portfolio_cols, facts_ff5)

    grs_rows.append({
        "set": name,
        "GRS_F": F_stat,
        "GRS_p": round(pVal, 4),
        "n_portfolios": len(portfolio_cols),
    })

ff5_grs_df = pd.DataFrame(grs_rows).set_index("set")
display(ff5_grs_df)

,GRS_F,GRS_p,n_portfolios
set,,,
SIZE,2.891426,0.0000,25
INV,3.420460,0.0000,25
OP,2.199372,0.0008,25


### Interpretation
FF5 achieves the lowest F-statistics across all portfolio sets (F = 2.89, 3.42, 2.20),
with the largest improvement on the investment and profitability sorts where RMW and CMA
are directly relevant. The GRS null is still rejected, consistent with Fama and French (2015)
who acknowledge rejection but argue the model provides an acceptable description of average returns.

# Conclusion and Summary
---
The GRS F-statistic can be seen below to decline monotonically with the addition of risk factors, over the three models.
The key takeaway is that each additional risk factor is reducing joint mispricing error, but not eliminating it entirely.

In [7]:
summary = pd.DataFrame({
    "CAPM": capm_grs_df["GRS_F"],
    "FF3": ff3_grs_df["GRS_F"],
    "FF5": ff5_grs_df["GRS_F"],
}).rename_axis("Portfolio Set")

display(summary)

,CAPM,FF3,FF5
Portfolio Set,,,
SIZE,4.452366,3.545125,2.891426
INV,5.513548,4.685143,3.420460
OP,2.413719,2.541858,2.199372


## Interpretation
---

**Market Factor (CAPM)**
CAPM performs the worst, with the investment portfolios showing the highest joint mispricing
(F = 5.51). We conclude that a single market factor is inadequate to explain cross-sectional
variation in returns

**Size and Value Factors (FF3)**
Adding SMB and HML reduces the F-statistic across all sets. However, FF3
performs relatively worse on the investment (F = 4.69) and profitability sets, as these
risk dimensions are absent from the model.

**Profitability and Investment Factors (FF5)**
FF5 achieves the largest reduction in joint pricing error, particularly on the investment (F = 3.43)
 and profitability sets (F = 2.20), where RMW and CMA are directly relevant.
  The size portfolio set remains the hardest to price across all models.

### Overall Conclusion
All three models are rejected by the GRS test, consistent with Fama and French (2015),
who acknowledge rejection but argue FF5 provides an acceptable description of average
returns. The monotonic decline in F-statistics from demonstrates meaningful improvement in pricing performance,
 with FF5 providing the best description of cross-sectional returns across all portfolio sets.